# 🎓 AI Impact on Students — Exploratory Data Analysis
### A comprehensive EDA of 50,000 students' AI usage, academic outcomes & well-being
---


## 0. 📦 Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# ── Style ────────────────────────────────────────────────────────────────────
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({
    "figure.dpi": 120,
    "figure.facecolor": "#0f0f0f",
    "axes.facecolor": "#1a1a2e",
    "axes.edgecolor": "#444",
    "axes.labelcolor": "white",
    "axes.titlecolor": "white",
    "xtick.color": "white",
    "ytick.color": "white",
    "text.color": "white",
    "grid.color": "#333",
    "legend.facecolor": "#1a1a2e",
    "legend.edgecolor": "#444",
    "legend.labelcolor": "white",
    "font.family": "DejaVu Sans",
})

ACCENT   = ["#7c83fd", "#fd7c7c", "#7cfd9e", "#fdd97c", "#fd7cdb"]
BURNOUT  = {"Low": "#7cfd9e", "Medium": "#fdd97c", "High": "#fd7c7c"}
POLICY   = {"Allowed_With_Citation":"#7c83fd","Strictly_Ban":"#fd7c7c","Actively_Encouraged":"#7cfd9e"}

print("✅ Setup complete")


✅ Setup complete


## 1. 📂 Load & Preview Data

In [2]:
df = pd.read_csv("//kaggle//input//datasets//laveshjadon//ai-impact-on-students")
# If running locally, change path above to your local path

print(f"Shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(10)


IsADirectoryError: [Errno 21] Is a directory: '//kaggle//input//datasets//laveshjadon//ai-impact-on-students'

## 2. 🔍 Data Overview

In [ ]:
# ── Data types & non-null counts ─────────────────────────────────────────────
print("=" * 55)
print("COLUMN INFO")
print("=" * 55)
df.info()


In [ ]:
# ── Missing values ───────────────────────────────────────────────────────────
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "✅ No missing values found!")


In [ ]:
# ── Statistical summary ──────────────────────────────────────────────────────
df.describe(include="all").T.style.background_gradient(cmap="Blues", axis=0)


In [ ]:
# ── Categorical value counts ─────────────────────────────────────────────────
cat_cols = df.select_dtypes(include="object").columns.tolist()
for col in cat_cols:
    print(f"\n{col}:")
    print(df[col].value_counts().to_string())


## 3. 📊 Univariate Analysis — Distributions

In [ ]:
# ── Numeric distributions ────────────────────────────────────────────────────
num_cols = ["Pre_Semester_GPA","Post_Semester_GPA","Weekly_GenAI_Hours",
            "Traditional_Study_Hours","Skill_Retention_Score",
            "Perceived_AI_Dependency","Anxiety_Level_During_Exams"]

fig, axes = plt.subplots(3, 3, figsize=(18, 13))
fig.suptitle("Numeric Feature Distributions", fontsize=16, fontweight="bold", y=1.01)
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    sns.histplot(df[col], bins=40, kde=True, ax=ax,
                 color=ACCENT[i % len(ACCENT)], edgecolor="none", alpha=0.8)
    ax.set_title(col.replace("_", " "), fontweight="bold")
    ax.set_xlabel("")

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
# ── Categorical distributions ────────────────────────────────────────────────
cat_display = ["Major_Category","Year_of_Study","Primary_Use_Case",
               "Prompt_Engineering_Skill","Institutional_Policy","Burnout_Risk_Level"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Categorical Feature Distributions", fontsize=16, fontweight="bold")
axes = axes.flatten()

for i, col in enumerate(cat_display):
    ax = axes[i]
    counts = df[col].value_counts()
    bars = ax.barh(counts.index, counts.values,
                   color=[ACCENT[j % len(ACCENT)] for j in range(len(counts))],
                   edgecolor="none", height=0.6)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
                f"{val:,}", va="center", ha="left", fontsize=9, color="white")
    ax.set_title(col.replace("_", " "), fontweight="bold")
    ax.set_xlabel("Count")
    ax.invert_yaxis()

plt.tight_layout()
plt.show()


## 4. 📈 GPA Analysis — Before vs After Semester

In [ ]:
# ── GPA change distribution ──────────────────────────────────────────────────
df["GPA_Change"] = df["Post_Semester_GPA"] - df["Pre_Semester_GPA"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("GPA Analysis", fontsize=15, fontweight="bold")

# Pre vs Post KDE
for col, c in zip(["Pre_Semester_GPA","Post_Semester_GPA"],["#7c83fd","#fd7c7c"]):
    sns.kdeplot(df[col], ax=axes[0], fill=True, alpha=0.5, label=col.replace("_"," "), color=c)
axes[0].set_title("Pre vs Post Semester GPA")
axes[0].legend()

# GPA Change histogram
sns.histplot(df["GPA_Change"], bins=50, kde=True, ax=axes[1],
             color="#7cfd9e", edgecolor="none")
axes[1].axvline(0, color="white", linestyle="--", alpha=0.6, label="No change")
axes[1].set_title("GPA Change Distribution")
axes[1].legend()

# GPA change by Major
major_gpa = df.groupby("Major_Category")["GPA_Change"].mean().sort_values()
colors_m = [ACCENT[i] for i in range(len(major_gpa))]
axes[2].barh(major_gpa.index, major_gpa.values, color=colors_m, edgecolor="none")
axes[2].axvline(0, color="white", linestyle="--", alpha=0.5)
axes[2].set_title("Avg GPA Change by Major")
axes[2].set_xlabel("Mean GPA Change")

plt.tight_layout()
plt.show()


In [ ]:
# ── GPA change by Year of Study ──────────────────────────────────────────────
order = ["Freshman","Sophomore","Junior","Senior","Graduate"]
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("GPA Change by Academic Level", fontsize=14, fontweight="bold")

sns.boxplot(data=df, x="Year_of_Study", y="GPA_Change",
            order=order, palette=ACCENT, ax=axes[0])
axes[0].axhline(0, color="white", linestyle="--", alpha=0.5)
axes[0].set_title("Distribution of GPA Change")

avg = df.groupby("Year_of_Study")["GPA_Change"].mean().reindex(order)
axes[1].bar(avg.index, avg.values,
            color=[ACCENT[i] for i in range(len(avg))], edgecolor="none")
axes[1].axhline(0, color="white", linestyle="--", alpha=0.5)
axes[1].set_title("Mean GPA Change by Year")
axes[1].set_ylabel("Mean GPA Change")

plt.tight_layout()
plt.show()


## 5. 🤖 AI Usage Patterns

In [ ]:
# ── AI hours vs GPA Change ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("AI Usage vs Academic Outcomes", fontsize=14, fontweight="bold")

# Scatter: AI hours vs GPA change (sampled for clarity)
sample = df.sample(3000, random_state=42)
sc = axes[0].scatter(sample["Weekly_GenAI_Hours"], sample["GPA_Change"],
                     c=sample["Skill_Retention_Score"], cmap="viridis",
                     alpha=0.5, s=15, edgecolors="none")
plt.colorbar(sc, ax=axes[0], label="Skill Retention Score")
axes[0].axhline(0, color="white", linestyle="--", alpha=0.4)
axes[0].set_xlabel("Weekly GenAI Hours")
axes[0].set_ylabel("GPA Change")
axes[0].set_title("AI Hours vs GPA Change
(colored by Skill Retention)")

# Binned average
df["AI_Bin"] = pd.cut(df["Weekly_GenAI_Hours"],
                      bins=[0,5,10,15,20,25,30,40],
                      labels=["0-5","5-10","10-15","15-20","20-25","25-30","30+"])
binned = df.groupby("AI_Bin", observed=True).agg(
    GPA_Change=("GPA_Change","mean"),
    Skill_Ret=("Skill_Retention_Score","mean")
).reset_index()

x = range(len(binned))
axes[1].bar(x, binned["GPA_Change"], color="#7c83fd", label="Avg GPA Change", alpha=0.85)
ax2 = axes[1].twinx()
ax2.plot(x, binned["Skill_Ret"], color="#fd7c7c", marker="o", label="Avg Skill Retention")
ax2.set_ylabel("Skill Retention Score", color="#fd7c7c")
ax2.tick_params(colors="#fd7c7c")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(binned["AI_Bin"].astype(str), rotation=30)
axes[1].set_xlabel("Weekly GenAI Hours")
axes[1].set_ylabel("Avg GPA Change", color="#7c83fd")
axes[1].tick_params(axis="y", colors="#7c83fd")
axes[1].set_title("AI Usage Bins vs GPA Change & Skill Retention")
axes[1].legend(loc="upper left")
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()


In [ ]:
# ── Prompt skill & Tool diversity ────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("AI Skill & Tool Usage", fontsize=14, fontweight="bold")

skill_order = ["Beginner","Intermediate","Advanced"]

# GPA by Prompt Skill
sns.boxplot(data=df, x="Prompt_Engineering_Skill", y="Post_Semester_GPA",
            order=skill_order, palette=ACCENT[:3], ax=axes[0])
axes[0].set_title("Post GPA by Prompt Skill Level")

# Skill retention by Prompt Skill
sns.violinplot(data=df, x="Prompt_Engineering_Skill", y="Skill_Retention_Score",
               order=skill_order, palette=ACCENT[:3], ax=axes[1], inner="quartile")
axes[1].set_title("Skill Retention by Prompt Skill Level")

# Tool diversity vs GPA
td = df.groupby("Tool_Diversity")["Post_Semester_GPA"].mean()
axes[2].plot(td.index, td.values, marker="o", color="#7c83fd", linewidth=2.5, markersize=8)
axes[2].fill_between(td.index, td.values, alpha=0.2, color="#7c83fd")
axes[2].set_xlabel("Number of AI Tools Used")
axes[2].set_ylabel("Avg Post-Semester GPA")
axes[2].set_title("Tool Diversity vs Post GPA")

plt.tight_layout()
plt.show()


## 6. 🧠 Student Well-being — Burnout, Anxiety & Dependency

In [ ]:
# ── Burnout risk breakdown ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Burnout Risk Analysis", fontsize=14, fontweight="bold")

# Overall pie
burnout_counts = df["Burnout_Risk_Level"].value_counts()
axes[0].pie(burnout_counts.values,
            labels=burnout_counts.index,
            colors=[BURNOUT[k] for k in burnout_counts.index],
            autopct="%1.1f%%", startangle=90,
            wedgeprops={"edgecolor":"#0f0f0f","linewidth":2})
axes[0].set_title("Overall Burnout Distribution")

# Burnout by Major
major_burnout = df.groupby(["Major_Category","Burnout_Risk_Level"]).size().unstack()
major_burnout_pct = major_burnout.div(major_burnout.sum(axis=1), axis=0) * 100
major_burnout_pct[["Low","Medium","High"]].plot(
    kind="bar", ax=axes[1], stacked=True,
    color=[BURNOUT["Low"], BURNOUT["Medium"], BURNOUT["High"]],
    edgecolor="none")
axes[1].set_title("Burnout Risk by Major")
axes[1].set_ylabel("Percentage (%)")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend(title="Risk Level", loc="upper right")

# Burnout by Institutional Policy
policy_burnout = df.groupby(["Institutional_Policy","Burnout_Risk_Level"]).size().unstack()
policy_burnout_pct = policy_burnout.div(policy_burnout.sum(axis=1), axis=0) * 100
policy_burnout_pct[["Low","Medium","High"]].plot(
    kind="bar", ax=axes[2], stacked=True,
    color=[BURNOUT["Low"], BURNOUT["Medium"], BURNOUT["High"]],
    edgecolor="none")
axes[2].set_title("Burnout Risk by Institutional Policy")
axes[2].set_ylabel("Percentage (%)")
axes[2].tick_params(axis="x", rotation=20)
axes[2].legend(title="Risk Level", loc="upper right")

plt.tight_layout()
plt.show()


In [ ]:
# ── Anxiety & Dependency ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Anxiety & AI Dependency Insights", fontsize=14, fontweight="bold")

# Anxiety by Burnout
sns.boxplot(data=df, x="Burnout_Risk_Level", y="Anxiety_Level_During_Exams",
            order=["Low","Medium","High"],
            palette=[BURNOUT["Low"],BURNOUT["Medium"],BURNOUT["High"]], ax=axes[0])
axes[0].set_title("Exam Anxiety by Burnout Risk")

# Dependency vs AI Hours
axes[1].scatter(df.sample(3000, random_state=1)["Perceived_AI_Dependency"],
                df.sample(3000, random_state=1)["Weekly_GenAI_Hours"],
                alpha=0.3, s=12, color="#7c83fd", edgecolors="none")
axes[1].set_xlabel("Perceived AI Dependency (1–10)")
axes[1].set_ylabel("Weekly GenAI Hours")
axes[1].set_title("AI Dependency vs Weekly AI Hours")

# Dependency by Burnout
sns.violinplot(data=df, x="Burnout_Risk_Level", y="Perceived_AI_Dependency",
               order=["Low","Medium","High"],
               palette=[BURNOUT["Low"],BURNOUT["Medium"],BURNOUT["High"]],
               ax=axes[2], inner="quartile")
axes[2].set_title("AI Dependency by Burnout Risk")

plt.tight_layout()
plt.show()


## 7. 🏛️ Institutional Policy Impact

In [ ]:
# ── Policy vs outcomes ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Institutional AI Policy vs Student Outcomes", fontsize=14, fontweight="bold")

policies = df["Institutional_Policy"].unique()
colors = [ACCENT[i] for i in range(len(policies))]

# Post GPA by policy
sns.boxplot(data=df, x="Institutional_Policy", y="Post_Semester_GPA",
            palette=ACCENT[:3], ax=axes[0])
axes[0].set_title("Post GPA by Policy")
axes[0].tick_params(axis="x", rotation=20)

# Skill retention by policy
sns.boxplot(data=df, x="Institutional_Policy", y="Skill_Retention_Score",
            palette=ACCENT[:3], ax=axes[1])
axes[1].set_title("Skill Retention by Policy")
axes[1].tick_params(axis="x", rotation=20)

# AI hours by policy
sns.boxplot(data=df, x="Institutional_Policy", y="Weekly_GenAI_Hours",
            palette=ACCENT[:3], ax=axes[2])
axes[2].set_title("Weekly AI Hours by Policy")
axes[2].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


## 8. 🔗 Correlation Analysis

In [ ]:
# ── Correlation heatmap ──────────────────────────────────────────────────────
num_df = df[["Pre_Semester_GPA","Post_Semester_GPA","Weekly_GenAI_Hours",
             "Traditional_Study_Hours","Tool_Diversity","Perceived_AI_Dependency",
             "Anxiety_Level_During_Exams","Skill_Retention_Score","GPA_Change"]]

corr = num_df.corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, linecolor="#222",
            annot_kws={"size": 9}, ax=ax)
ax.set_title("Correlation Matrix — Numeric Features", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()


In [ ]:
# ── Pairplot of key variables ────────────────────────────────────────────────
key_vars = ["Pre_Semester_GPA","Post_Semester_GPA",
            "Weekly_GenAI_Hours","Skill_Retention_Score","Burnout_Risk_Level"]
sample_pp = df[key_vars].sample(1500, random_state=42)

pp = sns.pairplot(sample_pp, hue="Burnout_Risk_Level",
                  palette=BURNOUT, plot_kws={"alpha":0.4, "s":15},
                  diag_kind="kde")
pp.fig.suptitle("Pairplot — Key Variables by Burnout Risk", y=1.01,
                fontsize=13, fontweight="bold")
plt.show()


## 9. 🎓 Major & Year-wise Deep Dive

In [ ]:
# ── Heatmap: Major × Year → Post GPA ─────────────────────────────────────────
pivot = df.pivot_table(values="Post_Semester_GPA",
                       index="Major_Category",
                       columns="Year_of_Study",
                       aggfunc="mean")[["Freshman","Sophomore","Junior","Senior","Graduate"]]

fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlOrRd",
            linewidths=0.5, linecolor="#222", ax=ax,
            cbar_kws={"label": "Avg Post GPA"})
ax.set_title("Avg Post-Semester GPA: Major × Year of Study",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ── AI hours & Traditional hours by Major ────────────────────────────────────
major_hours = df.groupby("Major_Category")[["Weekly_GenAI_Hours","Traditional_Study_Hours"]].mean()

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(major_hours))
w = 0.35
ax.bar(x - w/2, major_hours["Weekly_GenAI_Hours"],  width=w, label="AI Hours/Week",
       color="#7c83fd", edgecolor="none")
ax.bar(x + w/2, major_hours["Traditional_Study_Hours"], width=w, label="Traditional Hours/Week",
       color="#fd7c7c", edgecolor="none")
ax.set_xticks(x)
ax.set_xticklabels(major_hours.index)
ax.set_title("Average AI vs Traditional Study Hours by Major", fontweight="bold")
ax.set_ylabel("Hours per Week")
ax.legend()
plt.tight_layout()
plt.show()


## 10. 🛠️ Primary AI Use Case Analysis

In [ ]:
# ── Use case vs outcomes ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("AI Use Case vs Student Outcomes", fontsize=14, fontweight="bold")

use_order = df.groupby("Primary_Use_Case")["Post_Semester_GPA"].mean().sort_values().index

sns.barplot(data=df, x="Post_Semester_GPA", y="Primary_Use_Case",
            order=use_order, palette=ACCENT, ax=axes[0], orient="h", errorbar="sd")
axes[0].set_title("Post GPA by Use Case")

sns.barplot(data=df, x="Skill_Retention_Score", y="Primary_Use_Case",
            order=use_order, palette=ACCENT, ax=axes[1], orient="h", errorbar="sd")
axes[1].set_title("Skill Retention by Use Case")

sns.barplot(data=df, x="Perceived_AI_Dependency", y="Primary_Use_Case",
            order=use_order, palette=ACCENT, ax=axes[2], orient="h", errorbar="sd")
axes[2].set_title("AI Dependency by Use Case")

plt.tight_layout()
plt.show()


## 11. 💡 Key Insights Summary

| # | Insight |
|---|---------|
| 1 | **Moderate AI usage (5–15 hrs/week)** correlates with the best GPA outcomes; heavy usage (30+ hrs) tends to lower scores. |
| 2 | **Advanced prompt engineering skill** is associated with higher Post GPA and better skill retention. |
| 3 | **Higher AI dependency** strongly correlates with elevated burnout risk and exam anxiety. |
| 4 | **Actively Encouraged policy** institutions show higher weekly AI hours but mixed GPA outcomes. |
| 5 | **STEM students** use AI most for debugging; Humanities students lean toward copywriting/drafting. |
| 6 | **Graduate students** show the smallest GPA drop when using AI heavily, suggesting better self-regulation. |
| 7 | **Tool diversity (3–4 tools)** is associated with slightly better academic outcomes than using a single tool. |
| 8 | **High burnout risk** students show significantly higher exam anxiety and perceived AI dependency. |
| 9 | **Skill retention scores** decrease as weekly AI hours increase beyond 20 hrs, hinting at over-reliance. |
| 10 | **Paid subscription** holders tend to use more tools and have slightly higher prompt skill levels. |

---
*EDA complete. Next steps: Feature engineering → ML modelling → Burnout prediction & GPA regression.*
